# Primary cohort: discrimination against SIRS, horizons and conformal recertification

Runs the primary analyses on the common-support exact risk-set cohort: paired comparisons with SIRS, the 4- to 12-h horizons, the stacked ensemble in both directions and class-conditional conformal prediction with source, weighted and target calibration.

Data: PhysioNet/Computing in Cardiology Challenge 2019 training sets A and B (open access, CC BY 4.0), downloaded by the first code cell.

Run on Google Colab with a CPU runtime (Runtime > Run all). All results are written to `results_05.json`, which is downloaded at the end.

## Download the data

In [ ]:
import os, glob
!pip -q install awscli
!aws s3 sync --no-sign-request --only-show-errors s3://physionet-open/challenge-2019/1.0.0/training/ /content/data/
nA = len(glob.glob('/content/data/training_setA/*.psv')); nB = len(glob.glob('/content/data/training_setB/*.psv'))
print('Set A files:', nA, ' Set B files:', nB, ' (expected 20336 and 20000)')
if nA < 20336 or nB < 20000:
    print('S3 download incomplete - using wget instead (slower)')
    !wget -r -N -c -np -q https://physionet.org/files/challenge-2019/1.0.0/training/ -P /content/

## Load all ICU stays

In [ ]:
import os, glob, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
DATA = None
for cand in ['/content/data', '/content/physionet.org/files/challenge-2019/1.0.0/training']:
    if cand and glob.glob(f'{cand}/training_setA/*.psv'):
        DATA = cand; break
assert DATA, 'PhysioNet files not found - run the download cell first'
print('Using data in', DATA)
DYN = ['HR','O2Sat','Temp','SBP','MAP','DBP','Resp','EtCO2','BaseExcess','HCO3','FiO2','pH','PaCO2','SaO2',
       'AST','BUN','Alkalinephos','Calcium','Chloride','Creatinine','Bilirubin_direct','Glucose','Lactate',
       'Magnesium','Phosphate','Potassium','Bilirubin_total','TroponinI','Hct','Hgb','PTT','WBC','Fibrinogen','Platelets']
H = 6          # prediction horizon (hours)
W = 6          # feature window (hours)
W_MIN = 6      # minimum observed hours before a landmark
K_CTRL = 10    # controls per septic stay in risk-set sampling
SEEDS = [2026, 2027, 2028, 2029, 2030]
OUT = {}

def load_set(s):
    files = sorted(glob.glob(f'{DATA}/training_set{s}/*.psv'))
    stays = {}
    t0 = time.time()
    for f in files:
        d = pd.read_csv(f, sep='|')
        pid = os.path.basename(f)[:-4]
        lab = d['SepsisLabel'].to_numpy()
        pos = np.flatnonzero(lab == 1)
        stays[pid] = dict(X=d[DYN].to_numpy(np.float32), n=len(d),
                          first=(int(pos[0]) if len(pos) else -1),
                          age=float(d['Age'].iloc[0]), sex=float(d['Gender'].iloc[0]),
                          hadm=float(d['HospAdmTime'].iloc[0]), icu0=float(d['ICULOS'].iloc[0]),
                          unit=(0 if d['Unit1'].iloc[0] == 1 else (1 if d['Unit2'].iloc[0] == 1 else 2)))
    print(f'Set {s}: {len(stays)} stays loaded in {time.time()-t0:.0f}s')
    return stays

S = {s: load_set(s) for s in ['A', 'B']}
assert len(S['A']) > 0 and len(S['B']) > 0, 'Data not found'

## Helpers

In [ ]:
def features(st, lm):
    F = np.full((len(lm), 7 * len(DYN) + 4), np.nan, np.float32)
    meta = np.zeros((len(lm), 4), np.float32)   # age, sex, unit, tp (not model features)
    for r, (pid, y, tp) in enumerate(lm.itertuples(index=False)):
        v = st[pid]; hist = v['X'][:tp + 1]; win = hist[max(0, tp - W + 1):]
        with np.errstate(all='ignore'):
            ok = ~np.isnan(win); cnt = ok.sum(0)
            mean = np.nanmean(win, 0); sd = np.nanstd(win, 0); mn = np.nanmin(win, 0); mx = np.nanmax(win, 0)
            okh = ~np.isnan(hist)
            idx = np.where(okh.any(0), hist.shape[0] - 1 - np.argmax(okh[::-1], 0), -1)
            last = np.where(idx >= 0, hist[np.maximum(idx, 0), np.arange(hist.shape[1])], np.nan)
            t = np.arange(win.shape[0], dtype=np.float32)[:, None]
            tm = np.where(ok, t, 0).sum(0) / np.maximum(cnt, 1); vm = np.where(ok, win, 0).sum(0) / np.maximum(cnt, 1)
            cov = np.where(ok, (t - tm) * (np.nan_to_num(win) - vm), 0).sum(0)
            var = np.where(ok, (t - tm) ** 2, 0).sum(0)
            slope = np.where((cnt >= 2) & (var > 0), cov / np.where(var > 0, var, 1), np.nan)
            miss = 1 - cnt / win.shape[0]
        F[r, :7 * len(DYN)] = np.concatenate([mean, sd, mn, mx, last, slope, miss])
        F[r, -4:] = [v['age'], v['sex'], v['hadm'], miss.mean()]
        meta[r] = [v['age'], v['sex'], v['unit'], tp]
    return F, meta

def hgb(seed):
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                          max_depth=6, l2_regularization=1.0, class_weight='balanced',
                                          random_state=seed)

def wq_thresholds(cal_s, cal_w, test_w, alpha):
    """Weighted conformal quantile (Tibshirani et al. 2019) for each test point; inf if not attainable."""
    o = np.argsort(cal_s); s = cal_s[o]; cw = np.cumsum(cal_w[o]); tot = cw[-1]
    need = (1 - alpha) * (tot + test_w)
    k = np.searchsorted(cw, need, side='left')
    thr = np.full(len(test_w), np.inf); ok = k < len(s); thr[ok] = s[k[ok]]
    return thr

## Cohort builders

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def septic_cases(st, h):
    rows = []
    for pid, v in st.items():
        if v['first'] < 0: continue
        tp = v['first'] + 6 - h
        if W_MIN - 1 <= tp <= v['n'] - 1: rows.append((pid, 1, tp))
    return rows

def lm_trunc(st, h, seed):
    rng = np.random.default_rng(seed); cases = septic_cases(st, h); ref = np.array([c[2] for c in cases])
    rows = list(cases)
    for pid, v in st.items():
        if v['first'] >= 0: continue
        cand = ref[ref <= v['n'] - 1]
        if len(cand): rows.append((pid, 0, int(rng.choice(cand))))
    return pd.DataFrame(rows, columns=['pid', 'y', 'tp']), 0

def lm_riskset(st, h, seed, k):
    rng = np.random.default_rng(seed); cases = septic_cases(st, h)
    neg_pid = np.array([p for p, v in st.items() if v['first'] < 0]); neg_n = np.array([st[p]['n'] for p in neg_pid])
    rows, dropped = [], 0
    for pid, _, t in cases:
        pool = np.flatnonzero(neg_n - 1 >= t + h)
        if len(pool) < k: dropped += 1; continue
        rows.append((pid, 1, t)); rows += [(neg_pid[j], 0, t) for j in rng.choice(pool, size=k, replace=False)]
    return pd.DataFrame(rows, columns=['pid', 'y', 'tp']), dropped

def build(name, h, k=None):
    out = {}
    for s in ['A', 'B']:
        seed = 2019 + h + (0 if s == 'A' else 100)
        lm, dropped = lm_trunc(S[s], h, seed) if name == 'trunc' else lm_riskset(S[s], h, seed, k)
        X, meta = features(S[s], lm)
        out[s] = dict(X=X, y=lm.y.to_numpy(), tp=lm.tp.to_numpy(), pid=lm.pid.to_numpy(), dropped=int(dropped))
    return out

## Evaluation helpers

In [ ]:
NC = len(DYN)
VIT = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2']
LAB = [c for c in DYN if c not in VIT]
blk = lambda b, cols: [b * NC + DYN.index(c) for c in cols]
FAM = {'all': None,
       'vitals_values': sum((blk(b, VIT) for b in range(6)), []),
       'labs_values': sum((blk(b, LAB) for b in range(6)), []),
       'missingness_only': blk(6, DYN) + [7 * NC + 3],
       'demographics_only': [7 * NC, 7 * NC + 1, 7 * NC + 2]}

def last(X, c): return X[:, 4 * NC + DYN.index(c)]
def sirs(X):
    z = lambda c: np.nan_to_num(last(X, c), nan=-1e9)
    zt = last(X, 'Temp')
    return ((z('HR') > 90).astype(int) + (((zt > 38) | (zt < 36)) & ~np.isnan(zt)).astype(int)
            + ((z('Resp') > 20) | ((last(X, 'PaCO2') < 32) & ~np.isnan(last(X, 'PaCO2')))).astype(int)
            + (((last(X, 'WBC') > 12) | (last(X, 'WBC') < 4)) & ~np.isnan(last(X, 'WBC'))).astype(int)).astype(float)
def qsofa(X):
    rr = last(X, 'Resp'); sbp = last(X, 'SBP')
    return (((rr >= 22) & ~np.isnan(rr)).astype(int) + ((sbp <= 100) & ~np.isnan(sbp)).astype(int)).astype(float)

def stay_split(y, pid, frac, seed):
    rng = np.random.default_rng(seed); te = np.zeros(len(y), bool)
    for lab in (0, 1):
        u = np.unique(pid[y == lab]); pick = rng.choice(u, int(round(frac * len(u))), replace=False)
        te |= np.isin(pid, pick)
    # a control stay used in several risk sets must not appear on both sides
    te = np.isin(pid, np.unique(pid[te]))
    return ~te, te

def boot_auc(y, p, pid, B=200, seed=0):
    rng = np.random.default_rng(seed); code = pd.factorize(pid)[0]; order = np.argsort(code, kind='stable')
    bounds = np.r_[0, np.cumsum(np.bincount(code))]; n_u = len(bounds) - 1; a = []
    for _ in range(B):
        u = rng.integers(0, n_u, n_u); i = np.concatenate([order[bounds[j]:bounds[j + 1]] for j in u])
        if y[i].min() != y[i].max(): a.append(roc_auc_score(y[i], p[i]))
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]

def settings(C):
    """A->B, B->A transfer and A->A, B->B within-hospital (30% of stays held out)."""
    out = {}
    for src, tgt in (('A', 'B'), ('B', 'A')):
        out[f'{src}to{tgt}'] = (C[src]['X'], C[src]['y'], C[tgt]['X'], C[tgt]['y'], C[tgt]['pid'])
    for s in ('A', 'B'):
        tr, te = stay_split(C[s]['y'], C[s]['pid'], 0.3, 2026)
        out[f'{s}to{s}'] = (C[s]['X'][tr], C[s]['y'][tr], C[s]['X'][te], C[s]['y'][te], C[s]['pid'][te])
    return out

def fit_eval(Xtr, ytr, Xte, yte, cols=None, ci=None):
    keep = (~np.isnan(Xtr)).any(0) & (~np.isnan(Xte)).any(0)
    if cols is not None:
        m_ = np.zeros(Xtr.shape[1], bool); m_[cols] = True; keep &= m_
    m = hgb(2026).fit(Xtr[:, keep], ytr); p = m.predict_proba(Xte[:, keep])[:, 1]
    r = dict(AUROC=float(roc_auc_score(yte, p)), AUPRC=float(average_precision_score(yte, p)), prev=float(yte.mean()))
    if ci is not None: r['AUROC_CI'] = boot_auc(yte, p, ci)
    return r, m, keep

## Stacked pipeline definition

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
N_TREES = 200

def smote(X, y, seed, k=5):
    rng = np.random.default_rng(seed); Xp = X[y == 1]; n = int((y == 0).sum()) - int((y == 1).sum())
    if n <= 0: return X, y
    idx = NearestNeighbors(n_neighbors=k + 1).fit(Xp).kneighbors(Xp, return_distance=False)[:, 1:]
    i = rng.integers(0, len(Xp), n); j = idx[i, rng.integers(0, k, n)]; lam = rng.random((n, 1))
    return np.vstack([X, Xp[i] + lam * (Xp[j] - Xp[i])]), np.r_[y, np.ones(n, int)]

def bases(seed):
    return [hgb(seed).set_params(class_weight=None),
            RandomForestClassifier(n_estimators=N_TREES, min_samples_leaf=5, max_features='sqrt', n_jobs=-1, random_state=seed),
            ExtraTreesClassifier(n_estimators=N_TREES, min_samples_leaf=5, max_features='sqrt', n_jobs=-1, random_state=seed)]

def stacked_pipeline(Cs, Ct, seed=2026):
    X, y, pid = Cs['X'], Cs['y'], Cs['pid']
    # stay-level 60/20/20 split: train / isotonic / (conformal, unused here)
    rng = np.random.default_rng(seed); part = np.zeros(len(y), int)
    for lab in (0, 1):
        u = np.unique(pid[y == lab]); rng.shuffle(u); n = len(u)
        part[np.isin(pid, u[int(0.6 * n):int(0.8 * n)])] = 1; part[np.isin(pid, u[int(0.8 * n):])] = 2
    keep = (~np.isnan(X)).any(0) & (~np.isnan(Ct['X'])).any(0)
    tr = part == 0; ic = part == 1
    med = np.nanmedian(X[tr][:, keep], 0); med = np.nan_to_num(med, nan=0.0)
    f = lambda Z: np.nan_to_num(np.where(np.isnan(Z[:, keep]), med, Z[:, keep]))
    sc = StandardScaler().fit(f(X[tr])); P = lambda Z: sc.transform(f(Z))
    Xtr, Xic, Xt = P(X[tr]), P(X[ic]), P(Ct['X'])
    sel = np.argsort(-mutual_info_classif(Xtr, y[tr], random_state=seed))[:60]
    Xtr, Xic, Xt = Xtr[:, sel], Xic[:, sel], Xt[:, sel]
    ytr = y[tr]
    oof = np.zeros((len(ytr), 3))
    for a, b in StratifiedKFold(5, shuffle=True, random_state=seed).split(Xtr, ytr):
        Xa, ya = smote(Xtr[a], ytr[a], seed)
        for j, m in enumerate(bases(seed)): oof[b, j] = m.fit(Xa, ya).predict_proba(Xtr[b])[:, 1]
    meta = LogisticRegression(max_iter=1000).fit(oof, ytr)
    Xs_, ys_ = smote(Xtr, ytr, seed); ms = [m.fit(Xs_, ys_) for m in bases(seed)]
    S_ = lambda Z: meta.predict_proba(np.column_stack([m.predict_proba(Z)[:, 1] for m in ms]))[:, 1]
    iso = IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=1).fit(S_(Xic), y[ic])
    p_raw = S_(Xt); p = iso.predict(p_raw)
    return dict(AUROC_raw=float(roc_auc_score(Ct['y'], p_raw)), AUROC_isotonic=float(roc_auc_score(Ct['y'], p)),
                AUPRC_raw=float(average_precision_score(Ct['y'], p_raw)),
                AUROC_CI_raw=boot_auc(Ct['y'], p_raw, Ct['pid']), prev=float(Ct['y'].mean()))

## Primary cohorts, horizons 4/6/8/12 h

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
T_MAX = 48

def build_cs(name, h, k=10):
    out = {}
    for s in ['A', 'B']:
        seed = 2019 + h + (0 if s == 'A' else 100)
        sub = {p: v for p, v in S[s].items() if v['first'] < 0 or (v['first'] + 6 <= T_MAX)}
        lm, dropped = lm_trunc(sub, h, seed) if name == 'trunc' else lm_riskset(sub, h, seed, k)
        X, meta = features(S[s], lm)
        out[s] = dict(X=X, y=lm.y.to_numpy(), tp=lm.tp.to_numpy(), pid=lm.pid.to_numpy(), dropped=int(dropped))
    return out

PRIM = {h: build_cs('riskset', h) for h in (4, 6, 8, 12)}
ALL6 = build_cs('trunc', 6)          # all eligible stays (used only as a larger training set)
OUT['primary_cohorts'] = {str(h): {s: dict(rows=int(len(C[s]['y'])), septic=int(C[s]['y'].sum()), dropped=C[s]['dropped'],
                                          unique_control_stays=int(len(np.unique(C[s]['pid'][C[s]['y'] == 0]))),
                                          probe_tp_AUROC=float(roc_auc_score(C[s]['y'], C[s]['tp'])))
                                   for s in ('A', 'B')} for h, C in PRIM.items()}
print(json.dumps(OUT['primary_cohorts'], indent=1))

## Discrimination vs SIRS with paired stay-level bootstrap, 6 h

In [ ]:
def cluster_index(pid):
    code = pd.factorize(pid)[0]; order = np.argsort(code, kind='stable'); bounds = np.r_[0, np.cumsum(np.bincount(code))]
    return order, bounds

def paired_boot(y, scores, pid, ref='SIRS', B=500, seed=0):
    order, bounds = cluster_index(pid); n_u = len(bounds) - 1; rng = np.random.default_rng(seed)
    A_ = {k: [] for k in scores}; D_ = {k: [] for k in scores if k != ref}
    for _ in range(B):
        u = rng.integers(0, n_u, n_u); i = np.concatenate([order[bounds[j]:bounds[j + 1]] for j in u])
        if y[i].min() == y[i].max(): continue
        a = {k: roc_auc_score(y[i], s[i]) for k, s in scores.items()}
        for k in scores: A_[k].append(a[k])
        for k in D_: D_[k].append(a[k] - a[ref])
    res = {}
    for k, s in scores.items():
        res[k] = dict(AUROC=float(roc_auc_score(y, s)), AUPRC=float(average_precision_score(y, s)),
                      CI=[float(np.percentile(A_[k], 2.5)), float(np.percentile(A_[k], 97.5))])
        if k in D_:
            d = np.array(D_[k])
            res[k].update(delta_vs_SIRS=float(roc_auc_score(y, s) - roc_auc_score(y, scores[ref])),
                          delta_CI=[float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))],
                          p_two_sided=float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))))
    return res

def lr_scores(Xtr, ytr, Xte):
    keep = (~np.isnan(Xtr)).any(0) & (~np.isnan(Xte)).any(0)
    med = np.nan_to_num(np.nanmedian(Xtr[:, keep], 0), nan=0.0)
    f = lambda Z: np.nan_to_num(np.where(np.isnan(Z[:, keep]), med, Z[:, keep]))
    sc = StandardScaler().fit(f(Xtr))
    m = LogisticRegression(C=1.0, max_iter=3000, class_weight='balanced').fit(sc.transform(f(Xtr)), ytr)
    return m.predict_proba(sc.transform(f(Xte)))[:, 1]

def hgb_scores(Xtr, ytr, Xte):
    keep = (~np.isnan(Xtr)).any(0) & (~np.isnan(Xte)).any(0)
    return hgb(2026).fit(Xtr[:, keep], ytr).predict_proba(Xte[:, keep])[:, 1]

DISC = {}
C = PRIM[6]
for sname, (Xtr, ytr, Xte, yte, pte) in settings(C).items():
    sc = {'SIRS': sirs(Xte), 'qSOFA': qsofa(Xte),
          'Logistic regression': lr_scores(Xtr, ytr, Xte), 'Gradient boosting': hgb_scores(Xtr, ytr, Xte)}
    if sname in ('AtoB', 'BtoA'):
        src, tgt = sname[0], sname[-1]
        # gradient boosting trained on every eligible source stay (larger training set), scored on the risk-set target
        sc['Gradient boosting, all source stays'] = hgb_scores(ALL6[src]['X'], ALL6[src]['y'], Xte)
    DISC[sname] = paired_boot(yte, sc, pte)
    print(f"\n{sname}  (prevalence {yte.mean():.3f})")
    for k, v in DISC[sname].items():
        extra = f"  d vs SIRS {v['delta_vs_SIRS']:+.3f} [{v['delta_CI'][0]:+.3f},{v['delta_CI'][1]:+.3f}] p={v['p_two_sided']:.3f}" if 'delta_vs_SIRS' in v else ''
        print(f"  {k:38s} AUROC {v['AUROC']:.3f} [{v['CI'][0]:.3f}-{v['CI'][1]:.3f}]{extra}")
OUT['discrimination_6h'] = DISC

## Stacked pipeline on the primary cohort, both directions

In [ ]:
SUBP = {}
for src, tgt in (('A', 'B'), ('B', 'A')):
    t0 = time.time(); SUBP[f'{src}to{tgt}'] = stacked_pipeline(PRIM[6][src], PRIM[6][tgt])
    v = SUBP[f'{src}to{tgt}']
    print(f"stacked pipeline {src}->{tgt}: AUROC {v['AUROC_raw']:.3f} [{v['AUROC_CI_raw'][0]:.3f}-{v['AUROC_CI_raw'][1]:.3f}] ({time.time()-t0:.0f}s)")
OUT['stacked_pipeline_primary'] = SUBP

## Horizons 4/8/12 h

In [ ]:
HZ = {}
for h in (4, 8, 12):
    HZ[str(h)] = {}
    for sname, (Xtr, ytr, Xte, yte, pte) in settings(PRIM[h]).items():
        r = paired_boot(yte, {'SIRS': sirs(Xte), 'Gradient boosting': hgb_scores(Xtr, ytr, Xte)}, pte, B=300)
        HZ[str(h)][sname] = r
        g = r['Gradient boosting']
        print(f"h={h:2d} {sname}: GB {g['AUROC']:.3f} [{g['CI'][0]:.3f}-{g['CI'][1]:.3f}]  SIRS {r['SIRS']['AUROC']:.3f}  d {g['delta_vs_SIRS']:+.3f}")
OUT['horizons_primary'] = HZ

## Conformal: source-only, weighted, target label budget

In [ ]:
VITW = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp']
RESTRICT = [b * NC + DYN.index(c) for b in (0, 2, 3, 4) for c in VITW] + [7 * NC]
A1 = 0.10; A0S = (0.10, 0.20, 0.30); BUDGETS = [10, 20, 30, 50, 100]; DRAWS = 30; SEEDS = [2026, 2027, 2028, 2029, 2030]

def mondrian_thr(s1, s0, a1, a0):
    out = []
    for c, a in ((s1, a1), (s0, a0)):
        c = np.sort(c); k = int(np.ceil((len(c) + 1) * (1 - a))) - 1; out.append(c[k] if k < len(c) else np.inf)
    return out

def summarize(inc1, inc0, y):
    alert = inc1 & ~inc0; defer = inc1 & inc0
    return dict(septic_cov=float(inc1[y == 1].mean()), nonseptic_cov=float(inc0[y == 0].mean()),
                defer_rate=float(defer.mean()), alert_rate=float(alert.mean()), empty_rate=float((~inc1 & ~inc0).mean()),
                alert_PPV=(float(y[alert].mean()) if alert.any() else float('nan')))

def wq_thresholds(cal_s, cal_w, test_w, alpha):
    o = np.argsort(cal_s); s = cal_s[o]; cw = np.cumsum(cal_w[o]); tot = cw[-1]
    k = np.searchsorted(cw, (1 - alpha) * (tot + test_w), side='left')
    thr = np.full(len(test_w), np.inf); ok = k < len(s); thr[ok] = s[k[ok]]; return thr

def domain_weights(Xs, Xt, Xeval):
    allX = np.vstack([Xs, Xt]); med = np.nan_to_num(np.nanmedian(allX, 0), nan=0.0)
    f = lambda Z: np.nan_to_num(np.where(np.isnan(Z), med, Z), nan=0.0, posinf=0.0, neginf=0.0)
    sc = StandardScaler().fit(f(allX)); d = np.r_[np.zeros(len(Xs)), np.ones(len(Xt))]
    lr = LogisticRegression(C=0.1, max_iter=3000).fit(sc.transform(f(allX)), d)
    pe = np.clip(lr.predict_proba(sc.transform(f(Xeval)))[:, 1], 1e-6, 1 - 1e-6)
    return pe / (1 - pe) * len(Xs) / len(Xt), float(roc_auc_score(d, lr.predict_proba(sc.transform(f(allX)))[:, 1]))

def split_by_stay(y, pid, frac, seed):
    rng = np.random.default_rng(seed); u = np.unique(pid); pick = rng.choice(u, int(round(frac * len(u))), replace=False)
    cal = np.isin(pid, pick); return ~cal, cal

CONF = {}
for src, tgt in (('A', 'B'), ('B', 'A')):
    Cs, Ct = PRIM[6][src], PRIM[6][tgt]
    keep = (~np.isnan(Cs['X'])).any(0) & (~np.isnan(Ct['X'])).any(0)
    res = {}
    for seed in SEEDS:
        tr, cc = split_by_stay(Cs['y'], Cs['pid'], 0.3, seed)
        m = hgb(seed).fit(Cs['X'][tr][:, keep], Cs['y'][tr])
        pc = m.predict_proba(Cs['X'][cc][:, keep])[:, 1]; yc = Cs['y'][cc]
        pt = m.predict_proba(Ct['X'][:, keep])[:, 1]; yt = Ct['y']
        r = dict(AUROC=float(roc_auc_score(yt, pt)))
        for a0 in A0S:
            t1, t0 = mondrian_thr(1 - pc[yc == 1], pc[yc == 0], A1, a0)
            r[f'source_mondrian_a0_{a0}'] = summarize(1 - pt <= t1, pt <= t0, yt)
        half = np.random.default_rng(seed).random(len(yt)) < 0.5
        wt = np.zeros(len(yt)); wc = np.zeros(len(yc)); dauc = []
        for h_ in (half, ~half):
            w_e, a_ = domain_weights(Cs['X'][tr][:, RESTRICT], Ct['X'][~h_][:, RESTRICT],
                                     np.vstack([Ct['X'][h_][:, RESTRICT], Cs['X'][cc][:, RESTRICT]]))
            wt[h_] = w_e[:h_.sum()]; wc += 0.5 * w_e[h_.sum():]; dauc.append(a_)
        cap = np.quantile(np.r_[wc, wt], 0.99); wc = np.minimum(wc, cap); wt = np.minimum(wt, cap)
        r['restricted_domain_AUROC'] = float(np.mean(dauc))
        r['restricted_ESS_septic'] = float(wc[yc == 1].sum() ** 2 / (wc[yc == 1] ** 2).sum() / (yc == 1).sum())
        for a0 in A0S:
            t1w = wq_thresholds(1 - pc[yc == 1], wc[yc == 1], wt, A1); t0w = wq_thresholds(pc[yc == 0], wc[yc == 0], wt, a0)
            r[f'weighted_mondrian_a0_{a0}'] = summarize(1 - pt <= t1w, pt <= t0w, yt)
        tid = pd.factorize(Ct['pid'])[0]; rng = np.random.default_rng(seed + 7)
        pos_p = np.unique(tid[yt == 1]); neg_p = np.unique(tid[yt == 0])
        for nb in [b for b in BUDGETS if b <= len(pos_p) // 2 and 10 * b <= len(neg_p) // 2]:
            for a0 in A0S:
                draws = []
                for _ in range(DRAWS):
                    cp = rng.choice(pos_p, nb, replace=False); cn = rng.choice(neg_p, 10 * nb, replace=False)
                    calm = (np.isin(tid, cp) & (yt == 1)) | (np.isin(tid, cn) & (yt == 0)); evm = ~np.isin(tid, np.r_[cp, cn])
                    t1, t0 = mondrian_thr(1 - pt[calm & (yt == 1)], pt[calm & (yt == 0)], A1, a0)
                    draws.append(summarize(1 - pt[evm] <= t1, pt[evm] <= t0, yt[evm]))
                cov = np.array([d['septic_cov'] for d in draws])
                r[f'target_budget_{nb}_a0_{a0}'] = dict({k: float(np.nanmean([d[k] for d in draws])) for k in draws[0]},
                                                        septic_cov_p10=float(np.percentile(cov, 10)),
                                                        frac_draws_cov_ge_085=float(np.mean(cov >= 0.85)))
        res[seed] = r
    keys = [k for k in res[SEEDS[0]] if all(k in res[s_] for s_ in SEEDS)]
    agg = {}
    for k in keys:
        v0 = res[SEEDS[0]][k]
        if isinstance(v0, dict):
            agg[k] = {mm: [float(np.nanmean([res[s][k][mm] for s in SEEDS])), float(np.nanmin([res[s][k][mm] for s in SEEDS])),
                           float(np.nanmax([res[s][k][mm] for s in SEEDS]))] for mm in v0}
        else:
            agg[k] = [float(np.mean([res[s][k] for s in SEEDS])), float(np.min([res[s][k] for s in SEEDS])), float(np.max([res[s][k] for s in SEEDS]))]
    CONF[f'{src}to{tgt}'] = agg
    print(f"\nconformal {src}->{tgt}: AUROC {agg['AUROC'][0]:.3f}; restricted domain AUROC {agg['restricted_domain_AUROC'][0]:.3f}")
    for k in keys:
        if isinstance(agg[k], dict) and k.endswith('a0_0.1'):
            v = agg[k]
            print(f"  {k:32s} septic cov {v['septic_cov'][0]:.3f} [{v['septic_cov'][1]:.2f}-{v['septic_cov'][2]:.2f}]  "
                  f"defer {v['defer_rate'][0]:.3f}  alert {v['alert_rate'][0]:.3f}")
OUT['conformal_primary'] = CONF

## Save results

In [ ]:
with open('results_05.json', 'w') as fh:
    json.dump(OUT, fh, indent=1)
print('Saved results_05.json')

In [ ]:
from google.colab import files
files.download('results_05.json')